# Pipeline ETL — Projeto de Integração de Dados (Zika Vírus SINAN: 2024, 2025 e 2026)
**Universidade Federal de Pernambuco (UFPE) — Centro de Informática (CIn)**  
**Disciplina:** Banco de Dados 

---
## Parte 1: Extração, Diagnóstico de Qualidade e Transformação em Memória (`E` + `T1`)
Esta seção realiza a leitura das 3 bases anuais brutas (`ZIKABR24.csv`, `ZIKABR25.csv`, `ZIKABR26.csv`), evidencia os problemas estruturais entre os anos (prints intermediários de *Antes vs. Depois*) e entrega o DataFrame consolidado e higienizado (`df_zika_limpo`) pronto para a montagem do Esquema Estrela.


In [2]:
import pandas as pd
import numpy as np
import os

# Configuração de exibição do Pandas para prints de evidência
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)


### 1.1. Extração e Consolidação Inicial (`Extract`)
Carregamento dos arquivos CSV (localmente da pasta `../dados/` ou direto do repositório) e adição da coluna de rastreabilidade `ARQUIVO_ORIGEM`.


In [4]:
# Ajuste automático do caminho (funciona tanto na raiz quanto dentro de /notebooks no Colab)
caminho_base = '../dados/' if os.path.exists('../dados/ZIKABR24.csv') else ('dados/' if os.path.exists('dados/ZIKABR24.csv') else './')

zika_24 = pd.read_csv(os.path.join(caminho_base, 'ZIKABR24.csv'), low_memory=False)
zika_25 = pd.read_csv(os.path.join(caminho_base, 'ZIKABR25.csv'), low_memory=False)
zika_26 = pd.read_csv(os.path.join(caminho_base, 'ZIKABR26.csv'), low_memory=False)

zika_24['ARQUIVO_ORIGEM'] = 'ZIKABR24.csv'
zika_25['ARQUIVO_ORIGEM'] = 'ZIKABR25.csv'
zika_26['ARQUIVO_ORIGEM'] = 'ZIKABR26.csv'

df_bruto = pd.concat([zika_24, zika_25, zika_26], ignore_index=True)

print(f"Registros ZIKABR24: {len(zika_24):,}")
print(f"Registros ZIKABR25: {len(zika_25):,}")
print(f"Registros ZIKABR26: {len(zika_26):,}")
print(f"Total Consolidado Bruto: {len(df_bruto):,} linhas x {df_bruto.shape[1]} colunas")


Registros ZIKABR24: 43,312
Registros ZIKABR25: 25,764
Registros ZIKABR26: 13,384
Total Consolidado Bruto: 82,460 linhas x 39 colunas


### 1.2. Diagnóstico das Inconsistências nas Bases Brutas (Evidência Intermediária — *ANTES*)
Antes de aplicar as transformações, inspecionamos as divergências reais entre as safras 2024, 2025 e 2026 documentadas no dicionário do projeto.


In [5]:
print("=== [ANTES] 1. Divergência no código ID_AGRAVO ('A92.' truncado no ES em 2024 vs 'A928') ===")
print(pd.crosstab(df_bruto['ARQUIVO_ORIGEM'], df_bruto['ID_AGRAVO'], dropna=False))

print("\n=== [ANTES] 2. Tipagem mista e zeros falsos em COUFINF (UF de Infecção) ===")
print("Tipos Python encontrados em COUFINF:", df_bruto['COUFINF'].apply(lambda x: type(x).__name__).value_counts().to_dict())
print("Ocorrências da string 'ES' em COUFINF:", (df_bruto['COUFINF'] == 'ES').sum())
print("Ocorrências de 0 falso em COUFINF:", ((df_bruto['COUFINF'] == 0) | (df_bruto['COUFINF'] == '0')).sum())

print("\n=== [ANTES] 3. Vazamento de Ano (NU_ANO vs Arquivo de Origem) ===")
print(pd.crosstab(df_bruto['ARQUIVO_ORIGEM'], df_bruto['NU_ANO'], dropna=False))

print("\n=== [ANTES] 4. Amostra de NU_IDADE_N codificado (4 dígitos do SINAN) ===")
print(df_bruto[['ARQUIVO_ORIGEM', 'ANO_NASC', 'NU_IDADE_N']].head(5))


=== [ANTES] 1. Divergência no código ID_AGRAVO ('A92.' truncado no ES em 2024 vs 'A928') ===
ID_AGRAVO       A92.   A928
ARQUIVO_ORIGEM             
ZIKABR24.csv    5427  37885
ZIKABR25.csv       0  25764
ZIKABR26.csv       0  13384

=== [ANTES] 2. Tipagem mista e zeros falsos em COUFINF (UF de Infecção) ===
Tipos Python encontrados em COUFINF: {'float': 79331, 'str': 3129}
Ocorrências da string 'ES' em COUFINF: 12
Ocorrências de 0 falso em COUFINF: 1434

=== [ANTES] 3. Vazamento de Ano (NU_ANO vs Arquivo de Origem) ===
NU_ANO           2024   2025   2026
ARQUIVO_ORIGEM                     
ZIKABR24.csv    43243     69      0
ZIKABR25.csv        6  25752      6
ZIKABR26.csv        0      0  13384

=== [ANTES] 4. Amostra de NU_IDADE_N codificado (4 dígitos do SINAN) ===
  ARQUIVO_ORIGEM  ANO_NASC  NU_IDADE_N
0   ZIKABR24.csv    1990.0        4033
1   ZIKABR24.csv    2024.0        2001
2   ZIKABR24.csv    1993.0        4030
3   ZIKABR24.csv    1998.0        4025
4   ZIKABR24.csv    1995.

### 1.3. Transformação e Higienização (`Transform — T1`)
Aplicação sequencial das 6 regras de limpeza e padronização de negócio:
1. **Padronização de `ID_AGRAVO`:** Unificação de `'A92.'` (5.427 registros de 2024) para `'A928'`.
2. **Higienização de `COUFINF`:** Conversão da sigla textual `'ES'` para o código IBGE `32` e substituição de `0` falso por `NaN`.
3. **Eliminação de Zeros Falsos (`0 -> NaN`):** Conversão do preenchimento `0` (inexistente no dicionário SINAN) para `NaN` em `CLASSI_FIN`, `CRITERIO`, `EVOLUCAO`, `ID_RG_RESI`, `NDUPLIC_N`, `IN_VINCULA`, `ID_OCUPA_N`, `COMUNINF`, `DOENCA_TRA`, `TP_SISTEMA`, `TPUNINOT` e `COPAISINF` (preservando `0 = Analfabeto` em `CS_ESCOL_N`).
4. **Decodificação da Idade (`NU_IDADE_N`):** Conversão do código composto de 4 dígitos (`1xxx`=Horas, `2xxx`=Dias, `3xxx`=Meses, `4xxx`=Anos) para `idade_anos` e categorização em `faixa_etaria`.
5. **Saneamento de Datas e Cálculo de Métricas Temporais:** Conversão para `datetime`, cálculo de `qt_dias_atraso_notif` (`DT_NOTIFIC - DT_SIN_PRI`) e `qt_dias_encerramento` (`DT_ENCERRA - DT_NOTIFIC`), expurgando os 73 registros com intervalos negativos (`< 0` dias, como `-674`) ou superiores a 1 ano (`> 365` dias).
6. **Enriquecimento Semântico e Flags Analíticas:** Mapeamento de códigos IBGE de UF para siglas e regiões do Brasil, tradução dos domínios numéricos (`ds_sexo`, `ds_raca_cor`, `ds_gestante`, `ds_escolaridade`, `ds_classificacao_final`, `ds_criterio_confirmacao`, `ds_evolucao_caso`) e criação das flags binárias (`fl_confirmado`, `fl_descartado`, `fl_gestante`, `fl_obito`, `fl_autoctone`).


In [6]:
df_zika_limpo = df_bruto.copy()

# 1. Padronização de ID_AGRAVO
df_zika_limpo['ID_AGRAVO'] = df_zika_limpo['ID_AGRAVO'].astype(str).str.strip().replace({'A92.': 'A928'})

# 2. Dicionários Geográficos (IBGE <-> Sigla UF e Região)
uf_sigla_to_ibge = {
    'RO': 11, 'AC': 12, 'AM': 13, 'RR': 14, 'PA': 15, 'AP': 16, 'TO': 17,
    'MA': 21, 'PI': 22, 'CE': 23, 'RN': 24, 'PB': 25, 'PE': 26, 'AL': 27,
    'SE': 28, 'BA': 29, 'MG': 31, 'ES': 32, 'RJ': 33, 'SP': 35, 'PR': 41,
    'SC': 42, 'RS': 43, 'MS': 50, 'MT': 51, 'GO': 52, 'DF': 53
}
ibge_to_uf_sigla = {v: k for k, v in uf_sigla_to_ibge.items()}
ibge_to_regiao = {
    11:'Norte', 12:'Norte', 13:'Norte', 14:'Norte', 15:'Norte', 16:'Norte', 17:'Norte',
    21:'Nordeste', 22:'Nordeste', 23:'Nordeste', 24:'Nordeste', 25:'Nordeste', 26:'Nordeste', 27:'Nordeste', 28:'Nordeste', 29:'Nordeste',
    31:'Sudeste', 32:'Sudeste', 33:'Sudeste', 35:'Sudeste',
    41:'Sul', 42:'Sul', 43:'Sul',
    50:'Centro-Oeste', 51:'Centro-Oeste', 52:'Centro-Oeste', 53:'Centro-Oeste'
}

def limpar_coufinf(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().upper()
    if s in uf_sigla_to_ibge:
        return float(uf_sigla_to_ibge[s])
    try:
        num = float(s)
        return np.nan if num == 0.0 else num
    except:
        return np.nan

df_zika_limpo['COUFINF'] = df_zika_limpo['COUFINF'].apply(limpar_coufinf).astype('Int64')

# 3. Tratamento de Zeros Falsos (0 -> NaN)
colunas_zeros_falsos = [
    'CLASSI_FIN', 'CRITERIO', 'EVOLUCAO', 'ID_RG_RESI', 'NDUPLIC_N',
    'IN_VINCULA', 'ID_OCUPA_N', 'COMUNINF', 'DOENCA_TRA', 'TP_SISTEMA', 'TPUNINOT', 'COPAISINF'
]
for col in colunas_zeros_falsos:
    mask_zero = (df_zika_limpo[col] == 0) | (df_zika_limpo[col] == '0')
    df_zika_limpo.loc[mask_zero, col] = np.nan

# 4. Decodificação de NU_IDADE_N para idade em anos e faixa etária
def decodificar_idade_sinan(val):
    if pd.isna(val):
        return np.nan
    try:
        v = int(val)
        unidade, qtd = v // 1000, v % 1000
        if unidade == 4:
            return float(qtd) if qtd <= 120 else np.nan
        elif unidade == 3:
            return round(qtd / 12.0, 2)
        elif unidade == 2:
            return round(qtd / 365.0, 2)
        elif unidade == 1:
            return 0.0
        return np.nan
    except:
        return np.nan

def classificar_faixa_etaria(idade):
    if pd.isna(idade): return 'Ignorado'
    if idade < 1:      return '< 1 ano (Lactente)'
    if idade <= 9:     return '01 a 09 anos (Criança)'
    if idade <= 19:    return '10 a 19 anos (Adolescente)'
    if idade <= 39:    return '20 a 39 anos (Adulto Jovem)'
    if idade <= 59:    return '40 a 59 anos (Adulto)'
    return '60+ anos (Idoso)'

df_zika_limpo['idade_anos'] = df_zika_limpo['NU_IDADE_N'].apply(decodificar_idade_sinan)
df_zika_limpo['idade_anos_int'] = np.floor(df_zika_limpo['idade_anos']).astype('Int64')
df_zika_limpo['faixa_etaria'] = df_zika_limpo['idade_anos'].apply(classificar_faixa_etaria)

# 5. Conversão de Datas e Cálculo de Métricas Temporais Saneadas
colunas_data = ['DT_NOTIFIC', 'DT_SIN_PRI', 'DT_INVEST', 'DT_ENCERRA', 'DT_DIGITA', 'DT_OBITO']
for dcol in colunas_data:
    df_zika_limpo[dcol] = pd.to_datetime(df_zika_limpo[dcol], format='%Y-%m-%d', errors='coerce')

df_zika_limpo['ano_notificacao'] = df_zika_limpo['DT_NOTIFIC'].dt.year.astype('Int64')
df_zika_limpo['mes_notificacao'] = df_zika_limpo['DT_NOTIFIC'].dt.month.astype('Int64')
df_zika_limpo['trimestre_notificacao'] = df_zika_limpo['DT_NOTIFIC'].dt.quarter.astype('Int64')
df_zika_limpo['semana_epidemiologica'] = (df_zika_limpo['SEM_NOT'] % 100).astype('Int64')

atraso_dias = (df_zika_limpo['DT_NOTIFIC'] - df_zika_limpo['DT_SIN_PRI']).dt.days
encerra_dias = (df_zika_limpo['DT_ENCERRA'] - df_zika_limpo['DT_NOTIFIC']).dt.days

df_zika_limpo['qt_dias_atraso_notif'] = atraso_dias.where((atraso_dias >= 0) & (atraso_dias <= 365), np.nan)
df_zika_limpo['qt_dias_encerramento'] = encerra_dias.where((encerra_dias >= 0) & (encerra_dias <= 365), np.nan)

# 6. Decodificação de Domínios e Criação de Flags Analíticas
mapa_sexo = {'F': 'Feminino', 'M': 'Masculino', 'I': 'Ignorado'}
mapa_raca = {1: 'Branca', 2: 'Preta', 3: 'Amarela', 4: 'Parda', 5: 'Indígena', 9: 'Ignorado'}
mapa_gestante = {
    1: '1º Trimestre', 2: '2º Trimestre', 3: '3º Trimestre',
    4: 'Idade gestacional ignorada', 5: 'Não', 6: 'Não se aplica', 9: 'Ignorado'
}
mapa_escol = {
    0: 'Analfabeto', 1: '1ª a 4ª série incompleta', 2: '4ª série completa',
    3: '5ª à 8ª série incompleta', 4: 'Ensino fundamental completo',
    5: 'Ensino médio incompleto', 6: 'Ensino médio completo',
    7: 'Educação superior incompleta', 8: 'Educação superior completa',
    9: 'Ignorado', 10: 'Não se aplica'
}
mapa_classi = {1: 'Confirmado', 2: 'Descartado', 8: 'Inconclusivo'}
mapa_criterio = {1: 'Laboratorial', 2: 'Clínico-Epidemiológico'}
mapa_evolucao = {1: 'Cura', 2: 'Óbito pelo agravo', 3: 'Óbito por outras causas', 9: 'Ignorado'}

df_zika_limpo['ds_sexo'] = df_zika_limpo['CS_SEXO'].map(mapa_sexo).fillna('Ignorado')
df_zika_limpo['ds_raca_cor'] = df_zika_limpo['CS_RACA'].map(mapa_raca).fillna('Ignorado')
df_zika_limpo['ds_gestante'] = df_zika_limpo['CS_GESTANT'].map(mapa_gestante).fillna('Ignorado')
df_zika_limpo['ds_escolaridade'] = df_zika_limpo['CS_ESCOL_N'].map(mapa_escol).fillna('Ignorado')
df_zika_limpo['ds_classificacao_final'] = df_zika_limpo['CLASSI_FIN'].map(mapa_classi).fillna('Em Investigação / Ignorado')
df_zika_limpo['ds_criterio_confirmacao'] = df_zika_limpo['CRITERIO'].map(mapa_criterio).fillna('Não Informado')
df_zika_limpo['ds_evolucao_caso'] = df_zika_limpo['EVOLUCAO'].map(mapa_evolucao).fillna('Ignorado')

# Padronização de UF e Município (preenchendo 32 casos sem UF de residência pela UF de notificação)
df_zika_limpo['SG_UF_NOT'] = df_zika_limpo['SG_UF_NOT'].astype('Int64')
df_zika_limpo['SG_UF'] = df_zika_limpo['SG_UF'].fillna(df_zika_limpo['SG_UF_NOT']).astype('Int64')
df_zika_limpo['ID_MN_RESI'] = df_zika_limpo['ID_MN_RESI'].fillna(df_zika_limpo['ID_MUNICIP']).astype('Int64')

df_zika_limpo['sigla_uf_notificacao'] = df_zika_limpo['SG_UF_NOT'].map(ibge_to_uf_sigla)
df_zika_limpo['regiao_notificacao'] = df_zika_limpo['SG_UF_NOT'].map(ibge_to_regiao)
df_zika_limpo['sigla_uf_residencia'] = df_zika_limpo['SG_UF'].map(ibge_to_uf_sigla)
df_zika_limpo['regiao_residencia'] = df_zika_limpo['SG_UF'].map(ibge_to_regiao)

# Flags Binárias para a Tabela Fato
df_zika_limpo['fl_confirmado'] = (df_zika_limpo['CLASSI_FIN'] == 1).astype(int)
df_zika_limpo['fl_descartado'] = (df_zika_limpo['CLASSI_FIN'] == 2).astype(int)
df_zika_limpo['fl_gestante'] = df_zika_limpo['CS_GESTANT'].isin([1, 2, 3, 4]).astype(int)
df_zika_limpo['fl_obito'] = df_zika_limpo['EVOLUCAO'].isin([2, 3]).astype(int)
df_zika_limpo['fl_autoctone'] = (df_zika_limpo['TPAUTOCTO'] == 1).astype(int)

print("Transformação Parte 1 concluída! Shape de df_zika_limpo:", df_zika_limpo.shape)


Transformação Parte 1 concluída! Shape de df_zika_limpo: (82460, 64)


### 1.4. Validação Pós-Limpeza (Evidência Intermediária — *DEPOIS*)
Comprovação estatística das variáveis transformadas para inclusão no Relatório (Seção 4) e nos Slides.


In [7]:
print("=== [DEPOIS] 1. ID_AGRAVO 100% Padronizado ===")
print(df_zika_limpo['ID_AGRAVO'].value_counts(dropna=False))

print("\n=== [DEPOIS] 2. Estatísticas Descritivas das Métricas Contínuas Saneadas ===")
metricas = ['idade_anos', 'qt_dias_atraso_notif', 'qt_dias_encerramento']
resumo_stats = df_zika_limpo[metricas].agg(['count', 'mean', 'median', 'min', 'max', 'std']).T
print(resumo_stats.round(2))

print("\n=== [DEPOIS] 3. Distribuição por Faixa Etária Decodificada ===")
print(df_zika_limpo['faixa_etaria'].value_counts())

print("\n=== [DEPOIS] 4. Amostra das Colunas Prontas para o Esquema Estrela (Parte 2) ===")
colunas_contrato = [
    'DT_NOTIFIC', 'ano_notificacao', 'mes_notificacao', 'sigla_uf_residencia', 'ID_MN_RESI',
    'ds_sexo', 'idade_anos_int', 'faixa_etaria', 'ds_raca_cor', 'ds_gestante',
    'ds_classificacao_final', 'ds_criterio_confirmacao', 'ds_evolucao_caso',
    'qt_dias_atraso_notif', 'qt_dias_encerramento', 'fl_confirmado', 'fl_gestante', 'fl_obito'
]
print(df_zika_limpo[colunas_contrato].head(5))


=== [DEPOIS] 1. ID_AGRAVO 100% Padronizado ===
ID_AGRAVO
A928    82460
Name: count, dtype: int64

=== [DEPOIS] 2. Estatísticas Descritivas das Métricas Contínuas Saneadas ===
                        count   mean  median  min    max    std
idade_anos            82460.0  32.45    30.0  0.0  119.0  19.61
qt_dias_atraso_notif  82460.0   5.67     3.0  0.0  356.0  11.47
qt_dias_encerramento  80534.0  29.06    19.0  0.0  364.0  30.41

=== [DEPOIS] 3. Distribuição por Faixa Etária Decodificada ===
faixa_etaria
20 a 39 anos (Adulto Jovem)    31368
40 a 59 anos (Adulto)          19334
10 a 19 anos (Adolescente)     12597
60+ anos (Idoso)                8699
01 a 09 anos (Criança)          8254
< 1 ano (Lactente)              2208
Name: count, dtype: int64

=== [DEPOIS] 4. Amostra das Colunas Prontas para o Esquema Estrela (Parte 2) ===
  DT_NOTIFIC  ano_notificacao  mes_notificacao sigla_uf_residencia  ID_MN_RESI    ds_sexo  idade_anos_int                 faixa_etaria ds_raca_cor    ds_gestante 

---
## 🏛️ Parte 2: Modelagem Dimensional (Star Schema) e Carga no Banco (`T2` + `L`)
*(Espaço reservado para a Parte 2 da Dupla 1: utilizar o DataFrame `df_zika_limpo` acima para gerar as Surrogate Keys das 5 dimensões — `dim_tempo`, `dim_localidade`, `dim_unidade_notificadora`, `dim_paciente`, `dim_investigacao_desfecho` —, montar a `fato_notificacao_zika` e realizar a carga via `to_sql` no banco de dados).*
